# ML-08 — Capstone Modeling Lane: Predicting Content Decline

**Goal:** compare a learned classifier with the transparent Week-4 refresh rule using the same held-out clients and ranking metrics.

**Target:** `is_declining_label = 1` when `trend_direction == "down"`. `trend_direction` and `trend_pct` are target-derived and excluded from predictors. `content_id` and `client_id` are identifiers, not predictors.

**Evaluation unit:** content item. **Generalisation question:** can the model rank declining content for clients it did not see during training?

## 1. Method choice and why

I use a **Random Forest classifier** as the first nonlinear model. It can capture interactions between freshness, traffic, page characteristics, and categorical content metadata without requiring a linear relationship. A preprocessing pipeline handles numeric missing values and categorical fields consistently.

This is a ranking workflow: the team reviews a limited queue first. Therefore, the primary comparison is **Precision@20 and Precision@50**, not accuracy. These metrics answer: among the top 20 or 50 items flagged by each method, what fraction are actually labelled as declining? The model is not a causal explanation and should support review, not automatic content changes.

In [1]:
from pathlib import Path
import subprocess
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
REPO_URL = "https://github.com/Kifayat-Sayed/kifayat-flyrank-ml-internship.git"
ROOT = Path.cwd()
DATA_REL = Path("data/raw/content_refresh_anonymized.csv")

# Supports repo root, Colab clone, or an uploaded CSV in the current folder.
if (ROOT / DATA_REL).exists():
    DATA_PATH = ROOT / DATA_REL
elif (ROOT / "content_refresh_anonymized.csv").exists():
    DATA_PATH = ROOT / "content_refresh_anonymized.csv"
else:
    clone_dir = ROOT / "kifayat-flyrank-ml-internship"
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)], check=True)
    ROOT = clone_dir
    DATA_PATH = ROOT / DATA_REL

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find dataset at {DATA_PATH}")

data = pd.read_csv(DATA_PATH)
required = {"content_id", "client_id", "trend_direction", "trend_pct",
            "days_since_last_update", "impressions_90d"}
missing = required - set(data.columns)
if missing:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing)}")

data["is_declining_label"] = data["trend_direction"].astype(str).str.lower().eq("down").astype(int)
print(f"Loaded {len(data):,} rows × {data.shape[1]} columns")
print(f"Clients: {data['client_id'].nunique():,}")
print(f"Decline-label prevalence: {data['is_declining_label'].mean():.1%}")

Loaded 30,000 rows × 45 columns
Clients: 32
Decline-label prevalence: 54.2%


## 2. Split design

I use a deterministic **client-group holdout**. All rows belonging to a client stay together, so the test set represents clients unseen during training. This is more conservative than a random row split because pages from the same client can share patterns.

The split is created before model fitting. The Week-4 baseline rule and Random Forest are both scored on exactly the same test rows. No target-derived field (`trend_direction`, `trend_pct`, or the label) and no identifier (`content_id`, `client_id`) is used as a predictor.

In [2]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(splitter.split(data, data["is_declining_label"], groups=data["client_id"]))
train = data.iloc[train_idx].copy()
test = data.iloc[test_idx].copy()

train_clients = set(train["client_id"])
test_clients = set(test["client_id"])
assert train_clients.isdisjoint(test_clients), "Client leakage: a client appears in both splits."

print(f"Train: {len(train):,} rows | {train['client_id'].nunique():,} clients")
print(f"Test:  {len(test):,} rows | {test['client_id'].nunique():,} clients")
print(f"Train decline prevalence: {train['is_declining_label'].mean():.1%}")
print(f"Test decline prevalence:  {test['is_declining_label'].mean():.1%}")
print(f"Client overlap: {len(train_clients & test_clients)}")

Train: 23,837 rows | 25 clients
Test:  6,163 rows | 7 clients
Train decline prevalence: 55.0%
Test decline prevalence:  51.1%
Client overlap: 0


## 3. Train + compare against the Week-4 baseline

**Week-4 rule, reproduced exactly:** flag a page for refresh review when `days_since_last_update >= 180` and `impressions_90d >= 3000`; rank qualifying pages by impressions (non-qualifying pages receive score 0).

The Random Forest is trained only on the training clients. Both methods are evaluated on the held-out clients using Precision@20 and Precision@50. Ties in ranking are resolved by original row order, so results are reproducible.

In [3]:
TARGET = "is_declining_label"
EXCLUDED = {
    TARGET, "trend_direction", "trend_pct", "content_id", "client_id"
}
feature_cols = [col for col in data.columns if col not in EXCLUDED]

# Remove columns that are entirely missing in the training partition.
feature_cols = [col for col in feature_cols if not train[col].isna().all()]
X_train = train[feature_cols].copy()
X_test = test[feature_cols].copy()
y_train = train[TARGET].astype(int)
y_test = test[TARGET].astype(int)

numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = [col for col in X_train.columns if col not in numeric_features]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True))
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=10))
])
preprocess = ColumnTransformer([
    ("numeric", numeric_pipe, numeric_features),
    ("categorical", categorical_pipe, categorical_features)
])

model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", RandomForestClassifier(
        n_estimators=250,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=SEED
    ))
])
model.fit(X_train, y_train)
model_scores = model.predict_proba(X_test)[:, 1]

# Reproduce the transparent Week-4 baseline rule without using labels.
stale_days = pd.to_numeric(test["days_since_last_update"], errors="coerce")
impressions = pd.to_numeric(test["impressions_90d"], errors="coerce").fillna(0)
stale_flag = stale_days.ge(180).fillna(False).astype(int)
visible_flag = impressions.ge(3000).astype(int)
baseline_scores = stale_flag * visible_flag * impressions

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores), kind="stable")[:min(k, len(y_true))]
    return float(np.asarray(y_true)[order].mean())

def evaluate_ranking(name, scores):
    return {
        "Method": name,
        "Precision@20": precision_at_k(y_test, scores, 20),
        "Precision@50": precision_at_k(y_test, scores, 50),
        "Test items": len(y_test)
    }

comparison = pd.DataFrame([
    evaluate_ranking("Week-4 rule baseline", baseline_scores),
    evaluate_ranking("Random Forest", model_scores)
])
display(comparison.style.format({"Precision@20": "{:.1%}", "Precision@50": "{:.1%}"}))
print(f"Test label prevalence (reference only): {y_test.mean():.1%}")
print("Excluded from features:", sorted(EXCLUDED))
print(f"Predictor columns before encoding: {len(feature_cols)}")

,Method,Precision@20,Precision@50,Test items
0,Week-4 rule baseline,80.0%,70.0%,6163
1,Random Forest,100.0%,98.0%,6163


Test label prevalence (reference only): 51.1%
Excluded from features: ['client_id', 'content_id', 'is_declining_label', 'trend_direction', 'trend_pct']
Predictor columns before encoding: 40


## 4. Errors and interpretation

A ranking metric alone does not show what the model gets wrong. I inspect the model's top-ranked items and the threshold-0.50 classification errors. The top-ranked queue is intended for human review; a high score is not proof that a page is declining or that a refresh will help.

Permutation importance is calculated on held-out clients using ROC AUC as a diagnostic. It is a measure of predictive reliance, not causality. Correlated features can share or obscure importance.

In [4]:
# Threshold-based error review (separate from the ranking comparison above).
model_pred = (model_scores >= 0.50).astype(int)
print("Confusion matrix at threshold 0.50 [rows=true 0/1, columns=predicted 0/1]:")
print(confusion_matrix(y_test, model_pred, labels=[0, 1]))
print("\nClassification report:")
print(classification_report(y_test, model_pred, zero_division=0))

error_view = test[["content_id", "client_id", "trend_direction",
                   "days_since_last_update", "impressions_90d"]].copy()
error_view["actual_declining"] = y_test.to_numpy()
error_view["model_score"] = model_scores
error_view["predicted_declining_at_0_50"] = model_pred
error_view["error_type"] = np.select(
    [(error_view["actual_declining"].eq(0) & error_view["predicted_declining_at_0_50"].eq(1)),
     (error_view["actual_declining"].eq(1) & error_view["predicted_declining_at_0_50"].eq(0))],
    ["false positive", "false negative"],
    default="correct"
)
errors = error_view[error_view["error_type"] != "correct"].copy()
print(f"Misclassified test rows: {len(errors):,} / {len(test):,}")
display(errors.sort_values("model_score", ascending=False).head(10))

# Concrete examples of both error types, with identifiers retained only for internal debugging.
print("Examples: false positives (model predicts decline, label is not down)")
display(errors[errors["error_type"] == "false positive"].head(3))
print("Examples: false negatives (model misses labelled decline)")
display(errors[errors["error_type"] == "false negative"].sort_values("model_score").head(3))

Confusion matrix at threshold 0.50 [rows=true 0/1, columns=predicted 0/1]:
[[2246  768]
 [ 609 2540]]

Classification report:
              precision    recall  f1-score   support

           0       0.79      0.75      0.77      3014
           1       0.77      0.81      0.79      3149

    accuracy                           0.78      6163
   macro avg       0.78      0.78      0.78      6163
weighted avg       0.78      0.78      0.78      6163

Misclassified test rows: 1,377 / 6,163


,content_id,client_id,trend_direction,days_since_last_update,impressions_90d,actual_declining,model_score,predicted_declining_at_0_50,error_type
2357,content_8f1409b2674e,client_8527a891e2,stable,104,209,0,0.897177,1,false positive
12332,content_4d9f36001f06,client_8527a891e2,stable,104,3369,0,0.875538,1,false positive
20736,content_41baf0722ad9,client_8527a891e2,stable,104,3115,0,0.874437,1,false positive
5399,content_6677fd6c4ea5,client_4e07408562,stable,104,1152,0,0.854409,1,false positive
4050,content_500bd3907331,client_4e07408562,stable,104,4037,0,0.846480,1,false positive
10080,content_35d63627bf3e,client_8527a891e2,stable,103,1525,0,0.842661,1,false positive
22461,content_7e3be2e230f5,client_4e07408562,stable,104,909,0,0.830192,1,false positive
29456,content_b46c62b14582,client_8527a891e2,stable,103,6240,0,0.821281,1,false positive
28718,content_ef6e7d7cfe15,client_8527a891e2,stable,104,264,0,0.812232,1,false positive
11061,content_0b47dae0c7f9,client_8527a891e2,stable,103,1191,0,0.805765,1,false positive


Examples: false positives (model predicts decline, label is not down)


,content_id,client_id,trend_direction,days_since_last_update,impressions_90d,actual_declining,model_score,predicted_declining_at_0_50,error_type
13,content_a5a2fbc76336,client_8527a891e2,stable,103,307,0,0.724852,1,false positive
26,content_72c5c2d73e5a,client_4e07408562,stable,13,2426,0,0.661475,1,false positive
36,content_bce275871a25,client_f369cb89fc,stable,20,371,0,0.667557,1,false positive


Examples: false negatives (model misses labelled decline)


,content_id,client_id,trend_direction,days_since_last_update,impressions_90d,actual_declining,model_score,predicted_declining_at_0_50,error_type
24864,content_8ab9f1033843,client_e629fa6598,down,22,1265,1,0.196391,0,false negative
7744,content_b26ac45aca5d,client_4e07408562,down,7,2916,1,0.222382,0,false negative
1010,content_8ff857ae67d0,client_e629fa6598,down,22,21103,1,0.224568,0,false negative


In [5]:
from sklearn.metrics import roc_auc_score

# Use a modest deterministic sample for practical runtime on a notebook CPU.
n_perm = min(3000, len(X_test))
perm_idx = np.random.RandomState(SEED).choice(len(X_test), size=n_perm, replace=False)
X_perm = X_test.iloc[perm_idx]
y_perm = y_test.iloc[perm_idx]

perm = permutation_importance(
    model, X_perm, y_perm,
    scoring="roc_auc",
    n_repeats=3,
    random_state=SEED,
    n_jobs=-1
)
importance = pd.DataFrame({
    "feature": feature_cols,
    "permutation_auc_drop_mean": perm.importances_mean,
    "permutation_auc_drop_std": perm.importances_std
}).sort_values("permutation_auc_drop_mean", ascending=False)
display(importance.head(15))

# Compare the two ranking approaches and print a plain-language reading.
best_row = comparison.sort_values(["Precision@50", "Precision@20"], ascending=False).iloc[0]
print(f"Best by Precision@50 (tie-break Precision@20): {best_row['Method']}")
for metric in ["Precision@20", "Precision@50"]:
    b = comparison.loc[comparison["Method"] == "Week-4 rule baseline", metric].iloc[0]
    m = comparison.loc[comparison["Method"] == "Random Forest", metric].iloc[0]
    print(f"{metric}: baseline={b:.1%}; Random Forest={m:.1%}; difference={(m-b)*100:+.1f} percentage points")

,feature,permutation_auc_drop_mean,permutation_auc_drop_std
23,impressions_prev_30d,0.249032,0.006834
20,impressions_last_30d,0.222395,0.010788
10,impressions_90d,0.015702,0.001087
21,clicks_last_30d,0.014578,0.001498
38,impression_tier,0.011552,0.001303
18,days_with_impressions,0.010516,0.001282
22,sessions_last_30d,0.004628,0.000476
33,ctr,0.003851,0.000600
11,clicks_90d,0.002610,0.000288
34,avg_position,0.002397,0.000496


Best by Precision@50 (tie-break Precision@20): Random Forest
Precision@20: baseline=80.0%; Random Forest=100.0%; difference=+20.0 percentage points
Precision@50: baseline=70.0%; Random Forest=98.0%; difference=+28.0 percentage points


## Self-check

- [x] Method choice, split design, baseline comparison, and error analysis are documented.
- [x] Grouped split keeps clients disjoint between train and test.
- [x] `trend_direction`, `trend_pct`, target label, `content_id`, and `client_id` are excluded from model features.
- [x] Baseline and model use the same held-out rows and Precision@20 / Precision@50.
- [x] Results are calculated by the notebook; no metric values are hard-coded.
- [ ] Run every cell from top to bottom in the repository/Colab environment and review all outputs before committing.
- [ ] Check that the data file is not committed and that the final claims match the printed held-out results.

**Limitations:** the label is an observed trend-direction proxy, not a causal outcome. Client holdout is more honest for unseen-client generalisation, but one split can be noisy. This evaluation does not prove that refresh actions will reverse decline.